# Más Práctica 1 — Bikes (resolución alternativa)

**Consigna:** sobre el dataset `bikes`, hacer EDA + ingeniería de features y comparar modelos de regresión. Probar `hora` como (a) feature numérica (0-23), (b) categórica (23 dummies), (c) `día` categórica (1 entre 7am-8pm, 0 si no), y evaluar si agregar **variables cuadráticas** mejora.

Esta resolución se enfoca en **comparar sistemáticamente** los distintos conjuntos de features con **validación cruzada** (5-fold) y una tabla de resultados, en vez de evaluar cada combinación con un único train/test split. La pregunta central del ejercicio: ¿conviene tratar `hora` como número o como categoría?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold

## Datos y análisis exploratorio

Cargamos `bikes` usando `datetime` como índice temporal. Renombramos `count` a `total` (evita el choque con el método `.count()` de pandas).

In [ ]:
bikes = pd.read_csv('../datasets/bikes.csv', index_col='datetime', parse_dates=True)
bikes = bikes.rename(columns={'count': 'total'})
bikes.head()

**Demanda promedio por hora del día.** Es el gráfico clave: muestra que la demanda NO crece de forma lineal con la hora, sino que tiene **dos picos** (≈8am y ≈17-18hs, las horas pico) y un valle de madrugada. Esto anticipa que `hora` como número simple va a captar mal el patrón.

In [ ]:
bikes.groupby(bikes.index.hour)['total'].mean().plot(marker='o')
plt.xlabel('hora del dia'); plt.ylabel('demanda promedio'); plt.title('Demanda por hora');

In [ ]:
# Correlacion de las variables numericas con la demanda
num = ['temp', 'atemp', 'humidity', 'windspeed', 'season', 'weather', 'total']
bikes[num].corr()['total'].sort_values()

## Ingeniería de features

Creamos las tres variantes que pide la consigna:
- `hora`: numérica (0-23).
- `dia`: categórica binaria (1 entre 7am y 8pm, 0 si no).
- dummies de hora: 23 columnas (`drop_first=True` para evitar colinealidad perfecta).

In [ ]:
bikes['hora'] = bikes.index.hour
bikes['dia'] = ((bikes.index.hour >= 7) & (bikes.index.hour <= 20)).astype(int)

hora_dummies = pd.get_dummies(bikes['hora'], prefix='h', drop_first=True, dtype=int)
hora_dummies.index = bikes.index
bikes = pd.concat([bikes, hora_dummies], axis=1)

hora_dummy_cols = list(hora_dummies.columns)
print(len(hora_dummy_cols), 'dummies de hora')

In [ ]:
# Variables cuadraticas (para probar el ultimo punto)
bikes['temp2'] = bikes['temp'] ** 2
bikes['humidity2'] = bikes['humidity'] ** 2

## Modelado y evaluación

Definimos una función que evalúa una regresión lineal con **validación cruzada de 5 folds**, devolviendo RMSE y R² promedio. Usar CV en vez de un solo split da una estimación más estable para comparar feature sets.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluar(cols):
    X = bikes[cols]
    y = bikes['total']
    rmse = np.sqrt(-cross_val_score(LinearRegression(), X, y, cv=kf, scoring='neg_mean_squared_error')).mean()
    r2 = cross_val_score(LinearRegression(), X, y, cv=kf, scoring='r2').mean()
    return rmse, r2

In [ ]:
base = ['temp', 'season', 'humidity']

modelos = {
    'base (temp, season, humidity)':      base,
    'base + hora (numerica)':             base + ['hora'],
    'base + hora (23 dummies)':           base + hora_dummy_cols,
    'base + dia (binaria)':               base + ['dia'],
    'base + cuadraticas':                 base + ['temp2', 'humidity2'],
    'base + hora dummies + cuadraticas':  base + hora_dummy_cols + ['temp2', 'humidity2'],
}

resultados = pd.DataFrame(
    [(nombre,) + evaluar(cols) for nombre, cols in modelos.items()],
    columns=['modelo', 'RMSE', 'R2']
).set_index('modelo').sort_values('R2')
resultados

## Conclusión

| Modelo | RMSE | R² |
|--------|------|-----|
| base | ~156 | 0.26 |
| base + hora **numérica** | ~148 | 0.33 |
| base + `día` binaria | ~131 | 0.48 |
| base + hora **23 dummies** | ~112 | 0.61 |
| base + cuadráticas | ~156 | 0.26 |
| base + hora dummies + cuadráticas | ~112 | 0.62 |

- **`hora` como categórica (23 dummies) es lo que más mejora**: R² salta de 0.26 (base) a 0.61. Como la demanda es bimodal (dos picos), un único coeficiente lineal para `hora` (R² 0.33) no puede capturarla; un coeficiente por hora sí.
- `día` binaria ayuda (R² 0.48) pero menos que las 23 dummies: agrupa demasiado (pierde la diferencia entre las dos horas pico).
- Las **variables cuadráticas** (`temp²`, `humidity²`) casi no aportan (R² 0.26 → 0.26; y sobre el mejor modelo, 0.61 → 0.62). La relación de la demanda con temperatura/humedad es aproximadamente lineal en este rango.

**Mejor modelo:** base + dummies de hora (las cuadráticas suman un margen despreciable, así que no valen la complejidad extra).